# Answers: 4b · Fingerprints and MolPAL's own models

Worked answers to the **Try it yourself** exercises at the end of
[`4b_fingerprints_and_molpal.ipynb`](../4b_fingerprints_and_molpal.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

About 5 minutes in total.

In [ ]:
import sys; sys.path.insert(0, "..")
import json, io, contextlib, time
import numpy as np, pandas as pd
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")
from tour import *
import run_experiment as rx
from molpal.featurizer import Featurizer, feature_matrix
from molpal.models.sklmodels import RFModel
register_mini(rx)
scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)
order = np.random.default_rng(0).permutation(len(y)); train, test = order[:420], order[420:]

def spearman(a, b):
    return pd.Series(a).corr(pd.Series(b), method="spearman")

def run(mode, *options, seed=42):
    out = new_run_dir()
    sys.argv = ["run_experiment.py", "--dataset", "MINI", "--mode", mode, "--acq", "greedy", *options,
                "--init-size", "210", "--batch-size", "210", "--n-rounds", "5", "--topk", "100",
                "--seed", str(seed), "--run-dir", str(out)]
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        rx.main()
    return json.loads((out / "history.json").read_text())[-1]["topk_recall"], time.time() - t0
print("ready")

---
## Exercise 1: Morgan fingerprints instead of atom-pair

In [ ]:
for kind in ["pair", "morgan"]:
    featurize = Featurizer(fingerprint=kind, radius=2, length=2048)
    F = np.array(feature_matrix(list(scores["smiles"]), featurize))
    rf = RFModel(model_seed=0)
    rf.train(list(scores["smiles"][train]), y[train], featurizer=featurize)
    mu, _ = rf.get_means_and_vars(list(F[test]))
    print(f"{kind:7s} fingerprint: {F.sum(1).mean():5.1f} slots on per molecule   random-forest test Spearman {spearman(mu, y[test]):.3f}")

**Answer.** Worse, clearly: Morgan fingerprints give a test Spearman of 0.589, against 0.749 for atom-pair, with the same random forest. That supports MolPAL's choice of atom-pair fingerprints for this kind of task. A likely reason, not proven here: Morgan fingerprints describe small *local* neighbourhoods around each atom, while atom-pair fingerprints record pairs of atoms and their distances across the *whole* molecule. Bigger molecules switch on more atom-pair slots (78 on average, against 46 for Morgan), so atom-pair fingerprints carry more information about size and overall shape, and notebook 1 showed size matters a lot for docking scores. The choice of description matters, even among simple fingerprints.

---
## Exercise 2: the random forest with UCB

In [ ]:
for acq in ["greedy", "ucb"]:
    finals = [run("molpal", "--model", "rf", "--acq", acq, seed=s)[0] for s in [42, 43, 44]]
    print(f"random forest, {acq:6s} final recall {np.mean(finals):.3f}   seeds: {finals}")

**Answer.** UCB makes the random forest **worse**: 0.610 against greedy's 0.677, and less consistent (seeds 0.57–0.67 against 0.67–0.68). A forest's uncertainty is how much its 100 trees disagree, and the trees disagree most on unusual molecules, which mostly turn out to be average binders. Spending part of each batch on them costs hits. It's the same lesson as notebook 4: with a reasonable model, trusting the best guesses beats exploring.

---
## Exercise 3: how many Uni-Mol2 campaigns fit in one MPN campaign?

We time a Uni-Mol2 campaign here. For the MPN, use the time from your own run of the lesson
(the `seconds per campaign` column). When this was written it was about 920 seconds.

In [ ]:
# Run it twice and keep the faster time: the FIRST campaign in a fresh notebook also pays one-time
# start-up costs (setting up the GPU, importing libraries), which would make it look slower.
uni_seconds = min(run("mve", "--surrogate", "single", "--backbone", "unimol2")[1] for _ in range(2))
mpn_seconds = 920            # <- replace with the number from YOUR run of notebook 4b
print(f"Uni-Mol2 campaign: {uni_seconds:.0f} s   MPN campaign: {mpn_seconds} s   ->  about {mpn_seconds / uni_seconds:.0f} Uni-Mol2 campaigns per MPN campaign")

**Answer.** Roughly **100–130**. A Uni-Mol2 campaign takes about 7 seconds, and the MPN took 726 s in the lesson run and about 920 s in an earlier test, so 100–130 Uni-Mol2 campaigns fit in one MPN campaign. In research terms, that's the difference between testing a hundred ideas in the time it takes to test one.

Notice why the cell times the Uni-Mol2 campaign **twice** and keeps the faster run. The first campaign in a fresh notebook took about 45 seconds, because it also paid one-time start-up costs (setting up the GPU, loading libraries). Timing only the first run would have made Uni-Mol2 look 6× slower than it is. **When you time code, warm it up first.** Your exact numbers will differ with the hardware and how busy it is.